# Can we predict failure within 90 days of an inspection?
Target: `failed_90d`. Plan: inspect data → check for leakage → time-ordered split → baseline vs. gradient boosting → honest evaluation.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, roc_curve
from sklearn.inspection import permutation_importance
pd.set_option('display.width',140)
df=pd.read_csv('data.csv',parse_dates=['inspected_at']).sort_values('inspected_at').reset_index(drop=True)
print(df.shape, df.isna().sum().sum(),'missing;', df.asset_tag.nunique(),'unique assets;', df.duplicated().sum(),'duplicate rows')
print('base rate:',df.failed_90d.mean().round(4))
df.describe().T

## Failure rate over time
Inspections near the end of the data may not have a full 90-day follow-up window; a falling rate there would indicate censored labels.

In [ ]:
m=df.set_index('inspected_at').failed_90d.resample('MS').agg(['mean','count'])
m['mean'].plot(marker='o',title='Monthly failure rate'); plt.ylim(0,None); plt.show()
m.tail(6)

## Leakage check: `parts_replaced`
The dictionary says it counts parts replaced on the work order that *closed* the inspection record, which is information that may only exist after the outcome is known.

In [ ]:
df.groupby('parts_replaced').failed_90d.agg(['mean','count'])

If zero parts replaced means (almost) never failed and 2+ parts means always failed, the relationship is near-deterministic, which is implausible as a genuine pre-failure signal. It looks like repairs after a failure are recorded on the work order. **We exclude `parts_replaced`** since it wouldn't be available at inspection time. (Worth confirming with the data owner; the inflated score is shown below for contrast.)

## Other features vs. target

In [ ]:
feats=['vibration_rms','bearing_temp_c','ambient_temp_c','meter_hours','load_pct']
fig,ax=plt.subplots(1,5,figsize=(18,3))
for a,f in zip(ax,feats):
    q=pd.qcut(df[f],10,duplicates='drop'); df.groupby(q,observed=True).failed_90d.mean().reset_index(drop=True).plot(ax=a,marker='o'); a.set_title(f); a.set_xlabel('decile')
plt.tight_layout(); plt.show()
df.groupby('site').failed_90d.agg(['mean','count'])

## Modeling
Each asset appears once, so there's no asset-level leakage across rows. Because deployment means predicting the future, we hold out the **last 25% of inspections by time** as a test set, and tune nothing on it.

In [ ]:
cut=int(len(df)*.75); tr,te=df.iloc[:cut],df.iloc[cut:]
print('train',tr.inspected_at.min().date(),'->',tr.inspected_at.max().date(),'rate',tr.failed_90d.mean().round(3))
print('test ',te.inspected_at.min().date(),'->',te.inspected_at.max().date(),'rate',te.failed_90d.mean().round(3))
def models(num):
    return {'logistic':make_pipeline(ColumnTransformer([('n',StandardScaler(),num),('s',OneHotEncoder(),['site'])]),LogisticRegression(max_iter=1000)),
            'hist_gb':make_pipeline(ColumnTransformer([('n','passthrough',num),('s',OneHotEncoder(),['site'])]),
                                    HistGradientBoostingClassifier(max_depth=3,learning_rate=.05,max_iter=200,random_state=0))}
rows=[];fitted={}
for k,mod in models(feats).items():
    mod.fit(tr[feats+['site']],tr.failed_90d); p=mod.predict_proba(te[feats+['site']])[:,1]
    fitted[k]=(mod,p); rows.append(('no parts_replaced',k,roc_auc_score(te.failed_90d,p),average_precision_score(te.failed_90d,p)))
numL=feats+['parts_replaced']
for k,mod in models(numL).items():
    mod.fit(tr[numL+['site']],tr.failed_90d); p=mod.predict_proba(te[numL+['site']])[:,1]
    rows.append(('LEAKY (with parts_replaced)',k,roc_auc_score(te.failed_90d,p),average_precision_score(te.failed_90d,p)))
rows.append(('baseline','random',.5,te.failed_90d.mean()))
pd.DataFrame(rows,columns=['features','model','ROC AUC','PR AUC (avg precision)']).round(3)

The leaky variant looks near-perfect, which is exactly why it shouldn't be trusted. Below: the honest model's behaviour.

In [ ]:
y=te.failed_90d.values
fig,ax=plt.subplots(1,2,figsize=(11,4))
for k,(m_,p) in fitted.items():
    ax[0].plot(*roc_curve(y,p)[:2],label=k); pr,rc,_=precision_recall_curve(y,p); ax[1].plot(rc,pr,label=k)
ax[0].plot([0,1],[0,1],'k:'); ax[0].set(title='ROC',xlabel='FPR',ylabel='TPR')
ax[1].axhline(y.mean(),color='k',ls=':'); ax[1].set(title='Precision-Recall',xlabel='Recall',ylabel='Precision'); ax[0].legend(); plt.show()
rng=np.random.default_rng(0)
for k in fitted:
    p=fitted[k][1]; b=[roc_auc_score(y[i],p[i]) for i in (rng.integers(0,len(y),len(y)) for _ in range(500))]
    print(k,'AUC 95%% CI: %.3f-%.3f'%tuple(np.percentile(b,[2.5,97.5])))
t=pd.DataFrame({'y':y,'p':fitted['logistic'][1]}); t['bucket']=pd.qcut(t.p.rank(method='first'),5,labels=False)
print('Failure rate by risk quintile (logistic; 4 = highest risk):'); print(t.groupby('bucket').y.mean().round(3))

In [ ]:
mod=fitted['logistic'][0]; cols=feats+['site']
r=permutation_importance(mod,te[cols],y,scoring='roc_auc',n_repeats=20,random_state=0)
pd.Series(r.importances_mean,index=cols).sort_values().plot.barh(title='Permutation importance (drop in AUC), logistic'); plt.show()
names=mod[0].get_feature_names_out(); print(pd.Series(mod[-1].coef_[0],index=names).round(2).sort_values())

## Robustness checks
1. The monthly failure rate swings from ~1% to ~39% while feature means stay flat, so failures cluster in calendar time for reasons the features don't capture. A random split would mix periods and look different from the forward-looking test above.
2. `meter_hours` rises steadily with calendar date (assets age), so it can act as a clock. Test-period values lie beyond the training range, so check how much the result depends on it.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
X=df[feats+['site']]; Y=df.failed_90d
p=cross_val_predict(models(feats)['hist_gb'],X,Y,cv=StratifiedKFold(5,shuffle=True,random_state=0),method='predict_proba')[:,1]
print('random 5-fold CV AUC (hist_gb): %.3f   [time-ordered test: %.3f]'%(roc_auc_score(Y,p),roc_auc_score(y,fitted['hist_gb'][1])))
nm=[f for f in feats if f!='meter_hours']
for k,mod in models(nm).items():
    mod.fit(tr[nm+['site']],tr.failed_90d); print('time split, no meter_hours, %s AUC: %.3f'%(k,roc_auc_score(y,mod.predict_proba(te[nm+['site']])[:,1])))
# mean predicted risk vs actual rate in the test period (calibration under drift)
print('test: mean predicted %.3f vs actual %.3f'%(fitted['hist_gb'][1].mean(),y.mean()))

## Conclusions
- **Yes, but only modestly.** On a forward-in-time holdout, using only information available at inspection, gradient boosting reaches ROC AUC ≈ 0.75 (95% CI ≈ 0.69–0.81) and PR AUC ≈ 0.12 vs. 0.04 for chance. Logistic regression gets ≈ 0.66. Higher vibration, load and bearing temperature raise risk; site is uninformative.
- **`parts_replaced` is leakage** (0 parts → 0% failures, 2+ parts → 100%). Using it gives AUC ≈ 0.997, which is not a real result. Confirm with the data owner how that field is populated.
- **Calibration does not transfer across time.** Failure rate varies from ~1% to ~39% by month independent of the features; the model predicted 11% on average in the test period where the actual rate was 4%. Use scores for ranking/prioritisation, not as probabilities, unless recalibrated on recent data.
- **Time-split vs random-CV gap** (0.75 vs 0.85) shows random CV is optimistic, partly because `meter_hours` acts as a calendar proxy. Dropping it costs little (≈ 0.73).
- **Caveats:** the time-clustered failures suggest a driver not in the data (season, batch, operating events); I can't tell whether the late-2025 dip reflects real reliability or incomplete follow-up/labelling. Both are worth checking before deployment.